# PQ-02: Star-Schema Warehouse

## Objective
Build a warehouse from the PulseIQ source tables, validate dimension keys and fact-table relationships, reconcile row counts, and document the star schema.

## Data protection
The source DuckDB database will be opened in read-only mode.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# Locate the project and source database
PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "05_Starter" / "pulseiq.duckdb"

print("Project folder:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

assert DB_PATH.exists(), (
    "Database not found. Check that Jupyter is running from the pulseiq project folder."
)

# Connect without modifying the source database
con = duckdb.connect(str(DB_PATH), read_only=True)

# Confirm that the raw source tables are accessible
source_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""").fetchdf()

print("\nNumber of source tables:", len(source_tables))
display(source_tables)

Project folder: C:\Users\Hp\Documents\pulseiq
Database path: C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq.duckdb
Database exists: True

Number of source tables: 29


,table_name
0,budget_payroll_dept_month
1,data_contracts
2,data_dictionary
3,dim_account
4,dim_client
5,dim_date
6,dim_department
7,dim_employee
8,dim_leave_type
9,dim_location


In [2]:
# Inspect the employee dimension
employee_df = con.execute("""
    SELECT *
    FROM raw.dim_employee
    LIMIT 10
""").fetchdf()

print("Sample employee dimension records:")
display(employee_df)

print("\nEmployee dimension columns:")
print(employee_df.columns.tolist())

print("\nTotal employee dimension records:")
print(con.execute("""
    SELECT COUNT(*)
    FROM raw.dim_employee
""").fetchone()[0])

print("\nCurrent employee versions:")
print(con.execute("""
    SELECT COUNT(*)
    FROM raw.dim_employee
    WHERE is_current = TRUE
""").fetchone()[0])

Sample employee dimension records:


,employee_key,employee_id,full_name,gender,date_of_birth,hire_date,exit_date,employment_status,employment_type,department_id,...,role_title,job_level,manager_id,work_location,work_mode,highest_education,valid_from,valid_to,is_current,change_reason
0,100000,EMP-0001,Ganesh Murthy,Male,1989-10-08,2012-01-19,2020-01-24,Resigned,Permanent,D01,...,Associate Software Engineer,1,EMP-0051,Pune,Hybrid,B.E. Computer Science,2012-01-19,2012-05-12,N,Joining
1,100001,EMP-0001,Ganesh Murthy,Male,1989-10-08,2012-01-19,2020-01-24,Resigned,Permanent,D01,...,Associate Software Engineer,1,EMP-0002,Pune,Hybrid,B.E. Computer Science,2012-05-03,2015-05-13,N,Manager Change
2,100002,EMP-0001,Ganesh Murthy,Male,1989-10-08,2012-01-19,2020-01-24,Resigned,Permanent,D01,...,Associate Software Engineer,1,EMP-0002,Pune,Hybrid,B.E. Computer Science,2015-05-14,2020-01-24,Y,Manager Change
3,100003,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0051,Pune,Remote,B.E. Computer Science,2012-01-31,2016-03-29,N,Joining
4,100004,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0051,Pune,Remote,B.E. Computer Science,2016-03-30,2018-06-24,N,Manager Change
5,100005,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0078,Pune,Remote,B.E. Computer Science,2018-06-25,2021-05-21,N,Manager Change
6,100006,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0123,Pune,Remote,B.E. Computer Science,2021-05-22,2021-06-15,N,Manager Exit / Reassignment
7,100007,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0112,Pune,Remote,B.E. Computer Science,2021-06-16,2022-05-26,N,Manager Exit / Reassignment
8,100008,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D01,...,Tech Lead,4,EMP-0149,Pune,Remote,B.E. Computer Science,2022-05-27,2023-07-17,N,Manager Change
9,100009,EMP-0002,Jatin Jayaraman,Male,1973-06-25,2012-01-31,NaT,Active,Permanent,D08,...,Finance Manager,4,EMP-0361,Pune,Remote,B.E. Computer Science,2023-07-18,9999-12-31,Y,Transfer



Employee dimension columns:
['employee_key', 'employee_id', 'full_name', 'gender', 'date_of_birth', 'hire_date', 'exit_date', 'employment_status', 'employment_type', 'department_id', 'role_id', 'role_title', 'job_level', 'manager_id', 'work_location', 'work_mode', 'highest_education', 'valid_from', 'valid_to', 'is_current', 'change_reason']

Total employee dimension records:
1606

Current employee versions:
745


In [3]:
# Check employee dimension key quality
employee_key_checks = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT employee_key) AS distinct_employee_keys,
        COUNT(*) - COUNT(DISTINCT employee_key) AS duplicate_key_rows,
        COUNT(*) FILTER (WHERE employee_key IS NULL) AS null_employee_keys,
        COUNT(DISTINCT employee_id) AS distinct_employee_ids,
        COUNT(*) FILTER (WHERE is_current = 'Y') AS current_versions
    FROM raw.dim_employee
""").fetchdf()

display(employee_key_checks)

# Check how many employees have more than one historical version
multiple_versions = con.execute("""
    SELECT
        COUNT(*) AS employees_with_multiple_versions
    FROM (
        SELECT employee_id
        FROM raw.dim_employee
        GROUP BY employee_id
        HAVING COUNT(*) > 1
    ) AS employee_history
""").fetchdf()

display(multiple_versions)

,total_rows,distinct_employee_keys,duplicate_key_rows,null_employee_keys,distinct_employee_ids,current_versions
0,1606,1606,0,0,745,745


,employees_with_multiple_versions
0,389


In [4]:
# Basic employee dimension validation
print("Total rows:",
      con.execute("SELECT COUNT(*) FROM raw.dim_employee").fetchone()[0])

print("Distinct employee keys:",
      con.execute("""
          SELECT COUNT(DISTINCT employee_key)
          FROM raw.dim_employee
      """).fetchone()[0])

print("Null employee keys:",
      con.execute("""
          SELECT COUNT(*)
          FROM raw.dim_employee
          WHERE employee_key IS NULL
      """).fetchone()[0])

print("Distinct employee IDs:",
      con.execute("""
          SELECT COUNT(DISTINCT employee_id)
          FROM raw.dim_employee
      """).fetchone()[0])

print("Current versions:",
      con.execute("""
          SELECT COUNT(*)
          FROM raw.dim_employee
          WHERE is_current = 'Y'
      """).fetchone()[0])

Total rows: 1606
Distinct employee keys: 1606
Null employee keys: 0
Distinct employee IDs: 745
Current versions: 745


In [5]:
# Inspect the main fact-table join keys
join_key_map = {
    "fact_payroll": [
        "employee_key", "date_key", "department_id", "role_id"
    ],
    "fact_attendance_monthly": [
        "employee_key", "date_key", "department_id"
    ],
    "fact_attrition_risk": [
        "employee_key", "as_of_date_key", "department_id"
    ],
    "fact_leave": [
        "employee_key", "start_date_key", "end_date_key",
        "leave_type_code"
    ],
    "fact_ledger": [
        "date_key", "account_code", "cost_center_department_id"
    ],
    "fact_invoice": [
        "invoice_date_key", "client_id"
    ],
    "fact_invoice_payment": [
        "payment_date_key", "invoice_id"
    ],
    "fact_expense": [
        "expense_date_key", "vendor_id", "category_account"
    ],
    "fact_task": [
        "team_id", "sprint_id", "assignee_employee_id"
    ],
    "fact_forecast": [
        "department_id", "forecast_month_key"
    ],
    "fact_headcount_monthly": [
        "date_key", "department_id"
    ],
}

for table, keys in join_key_map.items():
    columns = set(
        con.execute(f"""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'raw'
              AND table_name = '{table}'
        """).fetchdf()["column_name"]
    )

    print(f"\n{table}")
    for key in keys:
        print(f"  {'✓' if key in columns else '✗'} {key}")


fact_payroll
  ✓ employee_key
  ✓ date_key
  ✓ department_id
  ✓ role_id

fact_attendance_monthly
  ✓ employee_key
  ✓ date_key
  ✓ department_id

fact_attrition_risk
  ✓ employee_key
  ✓ as_of_date_key
  ✓ department_id

fact_leave
  ✓ employee_key
  ✓ start_date_key
  ✓ end_date_key
  ✓ leave_type_code

fact_ledger
  ✓ date_key
  ✓ account_code
  ✓ cost_center_department_id

fact_invoice
  ✓ invoice_date_key
  ✓ client_id

fact_invoice_payment
  ✓ payment_date_key
  ✓ invoice_id

fact_expense
  ✓ expense_date_key
  ✓ vendor_id
  ✓ category_account

fact_task
  ✓ team_id
  ✓ sprint_id
  ✓ assignee_employee_id

fact_forecast
  ✓ department_id
  ✓ forecast_month_key

fact_headcount_monthly
  ✓ date_key
  ✓ department_id


In [6]:
# Validate foreign-key coverage against source dimensions

checks = {
    "Payroll -> Employee": """
        SELECT COUNT(*)
        FROM raw.fact_payroll f
        LEFT JOIN raw.dim_employee d
            ON f.employee_key = d.employee_key
        WHERE f.employee_key IS NOT NULL
          AND d.employee_key IS NULL
    """,
    "Payroll -> Date": """
        SELECT COUNT(*)
        FROM raw.fact_payroll f
        LEFT JOIN raw.dim_date d
            ON f.date_key = d.date_key
        WHERE f.date_key IS NOT NULL
          AND d.date_key IS NULL
    """,
    "Payroll -> Department": """
        SELECT COUNT(*)
        FROM raw.fact_payroll f
        LEFT JOIN raw.dim_department d
            ON f.department_id = d.department_id
        WHERE f.department_id IS NOT NULL
          AND d.department_id IS NULL
    """,
    "Attendance -> Employee": """
        SELECT COUNT(*)
        FROM raw.fact_attendance_monthly f
        LEFT JOIN raw.dim_employee d
            ON f.employee_key = d.employee_key
        WHERE f.employee_key IS NOT NULL
          AND d.employee_key IS NULL
    """,
    "Leave -> Employee": """
        SELECT COUNT(*)
        FROM raw.fact_leave f
        LEFT JOIN raw.dim_employee d
            ON f.employee_key = d.employee_key
        WHERE f.employee_key IS NOT NULL
          AND d.employee_key IS NULL
    """,
    "Ledger -> Account": """
        SELECT COUNT(*)
        FROM raw.fact_ledger f
        LEFT JOIN raw.dim_account d
            ON f.account_code = d.account_code
        WHERE f.account_code IS NOT NULL
          AND d.account_code IS NULL
    """,
    "Invoice -> Client": """
        SELECT COUNT(*)
        FROM raw.fact_invoice f
        LEFT JOIN raw.dim_client d
            ON f.client_id = d.client_id
        WHERE f.client_id IS NOT NULL
          AND d.client_id IS NULL
    """,
    "Expense -> Vendor": """
        SELECT COUNT(*)
        FROM raw.fact_expense f
        LEFT JOIN raw.dim_vendor d
            ON f.vendor_id = d.vendor_id
        WHERE f.vendor_id IS NOT NULL
          AND d.vendor_id IS NULL
    """,
    "Task -> Team": """
        SELECT COUNT(*)
        FROM raw.fact_task f
        LEFT JOIN raw.dim_team d
            ON f.team_id = d.team_id
        WHERE f.team_id IS NOT NULL
          AND d.team_id IS NULL
    """,
    "Task -> Sprint": """
        SELECT COUNT(*)
        FROM raw.fact_task f
        LEFT JOIN raw.dim_sprint d
            ON f.sprint_id = d.sprint_id
        WHERE f.sprint_id IS NOT NULL
          AND d.sprint_id IS NULL
    """
}

results = []

for relationship, sql in checks.items():
    orphan_count = con.execute(sql).fetchone()[0]
    results.append({
        "relationship": relationship,
        "orphan_rows": orphan_count,
        "status": "PASS" if orphan_count == 0 else "REVIEW"
    })

coverage_df = pd.DataFrame(results)
display(coverage_df)

,relationship,orphan_rows,status
0,Payroll -> Employee,2,REVIEW
1,Payroll -> Date,0,PASS
2,Payroll -> Department,0,PASS
3,Attendance -> Employee,0,PASS
4,Leave -> Employee,0,PASS
5,Ledger -> Account,0,PASS
6,Invoice -> Client,0,PASS
7,Expense -> Vendor,0,PASS
8,Task -> Team,0,PASS
9,Task -> Sprint,0,PASS


In [7]:
# Inspect payroll records with unmatched employee keys
orphan_payroll = con.execute("""
    SELECT
        f.payroll_item_id,
        f.employee_id,
        f.employee_key,
        f.period_end_date,
        f.gross_pay,
        f.net_pay
    FROM raw.fact_payroll f
    LEFT JOIN raw.dim_employee d
        ON f.employee_key = d.employee_key
    WHERE f.employee_key IS NOT NULL
      AND d.employee_key IS NULL
    ORDER BY f.employee_id
""").fetchdf()

print("Orphan payroll records:", len(orphan_payroll))
display(orphan_payroll)

Orphan payroll records: 2


,payroll_item_id,employee_id,employee_key,period_end_date,gross_pay,net_pay
0,PI-X0003,EMP-9001,-1,2026-04-30,96015.00,95815.00
1,PI-X0004,EMP-9002,-1,2025-12-31,96527.42,94527.42


In [8]:
# Check uniqueness and nulls in dimension keys

dimension_keys = {
    "dim_date": "date_key",
    "dim_department": "department_id",
    "dim_role": "role_id",
    "dim_account": "account_code",
    "dim_client": "client_id",
    "dim_leave_type": "leave_type_code",
    "dim_location": "work_location",
    "dim_sprint": "sprint_id",
    "dim_team": "team_id",
    "dim_vendor": "vendor_id",
}

dimension_results = []

for table, key in dimension_keys.items():
    row = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT("{key}") AS non_null_keys,
            COUNT(DISTINCT "{key}") AS distinct_keys
        FROM raw."{table}"
    """).fetchone()

    total_rows, non_null_keys, distinct_keys = row
    duplicate_rows = non_null_keys - distinct_keys
    null_keys = total_rows - non_null_keys

    dimension_results.append({
        "dimension": table,
        "key": key,
        "total_rows": total_rows,
        "null_keys": null_keys,
        "duplicate_key_rows": duplicate_rows,
        "status": (
            "PASS"
            if null_keys == 0 and duplicate_rows == 0
            else "REVIEW"
        )
    })

dimension_key_report = pd.DataFrame(dimension_results)
display(dimension_key_report)

,dimension,key,total_rows,null_keys,duplicate_key_rows,status
0,dim_date,date_key,2557,0,0,PASS
1,dim_department,department_id,11,0,0,PASS
2,dim_role,role_id,58,0,0,PASS
3,dim_account,account_code,47,0,0,PASS
4,dim_client,client_id,28,0,0,PASS
5,dim_leave_type,leave_type_code,6,0,0,PASS
6,dim_location,work_location,9,0,0,PASS
7,dim_sprint,sprint_id,38,0,0,PASS
8,dim_team,team_id,16,0,0,PASS
9,dim_vendor,vendor_id,30,0,0,PASS


In [9]:
schemas = con.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
""").fetchdf()

display(schemas)

,schema_name
0,information_schema
1,main
2,main
3,main
4,pg_catalog
5,raw


In [10]:
schema_names = con.execute("""
    SELECT DISTINCT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
""").fetchdf()

display(schema_names)

warehouse_exists = con.execute("""
    SELECT COUNT(*)
    FROM information_schema.schemata
    WHERE schema_name = 'warehouse'
""").fetchone()[0] > 0

print("Warehouse schema already exists:", warehouse_exists)

,schema_name
0,information_schema
1,main
2,pg_catalog
3,raw


Warehouse schema already exists: False


In [11]:
# Create a separate writable connection for warehouse objects
con_wh = duckdb.connect(str(DB_PATH), read_only=False)

# Create the warehouse schema without changing raw source tables
con_wh.execute("CREATE SCHEMA IF NOT EXISTS warehouse")

# Verify the schema exists
schemas = con_wh.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name IN ('raw', 'warehouse')
    ORDER BY schema_name
""").fetchdf()

display(schemas)

print("Warehouse schema created successfully.")

ConnectionException: Connection Error: Can't open a connection to same database file with a different configuration than existing connections

In [ ]:
# Close the existing read-only connection
con.close()

# Open one writable connection
con_wh = duckdb.connect(str(DB_PATH), read_only=False)

# Create a separate warehouse schema
con_wh.execute("CREATE SCHEMA IF NOT EXISTS warehouse")

# Verify the schemas
display(con_wh.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name IN ('raw', 'warehouse')
    ORDER BY schema_name
""").fetchdf())

print("Warehouse schema created successfully!")

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "05_Starter" / "pulseiq.duckdb"

print("Database exists:", DB_PATH.exists())

con_wh = duckdb.connect(str(DB_PATH), read_only=False)

con_wh.execute("CREATE SCHEMA IF NOT EXISTS warehouse")

print(
    con_wh.execute("""
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'warehouse')
        ORDER BY schema_name
    """).fetchdf()
)

print("Warehouse schema is ready!")

Database exists: True


IOException: IO Error: Cannot open file "C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\Hp\AppData\Local\Programs\Python\Python314\python.exe (PID 29568)

In [2]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()

SOURCE_DB = PROJECT_ROOT / "05_Starter" / "pulseiq.duckdb"
WAREHOUSE_DB = PROJECT_ROOT / "05_Starter" / "pulseiq_warehouse.duckdb"

print("Source database exists:", SOURCE_DB.exists())
print("Warehouse database already exists:", WAREHOUSE_DB.exists())

assert SOURCE_DB.exists(), "Source database not found."

# Create or open a separate warehouse database
con_wh = duckdb.connect(str(WAREHOUSE_DB))

# Attach the starter database for read-only access
source_path = str(SOURCE_DB).replace("\\", "/")
con_wh.execute(
    f"ATTACH '{source_path}' AS source_db (READ_ONLY)"
)

# Create the warehouse schema in the separate database
con_wh.execute("CREATE SCHEMA IF NOT EXISTS warehouse")

print("\nAttached source schemas:")
display(con_wh.execute("""
    SELECT schema_name
    FROM source_db.information_schema.schemata
    ORDER BY schema_name
""").fetchdf())

print("\nWarehouse database ready:", WAREHOUSE_DB)

Source database exists: True
Warehouse database already exists: False

Attached source schemas:


CatalogException: Catalog Error: Table with name "information_schema.schemata" does not exist because schema "information_schema" does not exist.

LINE 3:     FROM source_db.information_schema.schemata
                 ^

In [3]:
# List attached databases
display(con_wh.execute("PRAGMA database_list").fetchdf())

# Verify source tables are accessible
source_tables = con_wh.execute("""
    SELECT table_name
    FROM source_db.raw.information_schema.tables
    ORDER BY table_name
""").fetchdf()

print("Source table count:", len(source_tables))
display(source_tables)

# Verify the warehouse schema exists
warehouse_schemas = con_wh.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name = 'warehouse'
""").fetchdf()

print("\nWarehouse schema:")
display(warehouse_schemas)

,seq,name,file
0,20624,pulseiq_warehouse,C:\Users\Hp\Documents\pulseiq\05_Starter\pulse...
1,22227,source_db,C:\Users\Hp\Documents\pulseiq\05_Starter\pulse...


ParserException: Parser Error: NameListToString NOT IMPLEMENTED

In [4]:
# List tables from the attached source database
source_tables = con_wh.execute(
    "SHOW TABLES FROM source_db.raw"
).fetchdf()

print("Source table count:", len(source_tables))
display(source_tables)

Source table count: 29


,name
0,budget_payroll_dept_month
1,data_contracts
2,data_dictionary
3,dim_account
4,dim_client
5,dim_date
6,dim_department
7,dim_employee
8,dim_leave_type
9,dim_location


In [5]:
# Dimension tables to copy into the warehouse
dimension_tables = [
    "dim_account",
    "dim_client",
    "dim_date",
    "dim_department",
    "dim_employee",
    "dim_leave_type",
    "dim_location",
    "dim_role",
    "dim_sprint",
    "dim_team",
    "dim_vendor",
]

for table in dimension_tables:
    con_wh.execute(f"""
        CREATE TABLE warehouse.{table} AS
        SELECT *
        FROM source_db.raw.{table}
    """)
    print(f"Created warehouse.{table}")

print("\nDimension row-count validation:")

for table in dimension_tables:
    source_count = con_wh.execute(
        f"SELECT COUNT(*) FROM source_db.raw.{table}"
    ).fetchone()[0]

    warehouse_count = con_wh.execute(
        f"SELECT COUNT(*) FROM warehouse.{table}"
    ).fetchone()[0]

    status = "PASS" if source_count == warehouse_count else "FAIL"

    print(
        f"{table}: source={source_count:,}, "
        f"warehouse={warehouse_count:,} — {status}"
    )

Created warehouse.dim_account
Created warehouse.dim_client
Created warehouse.dim_date
Created warehouse.dim_department
Created warehouse.dim_employee
Created warehouse.dim_leave_type
Created warehouse.dim_location
Created warehouse.dim_role
Created warehouse.dim_sprint
Created warehouse.dim_team
Created warehouse.dim_vendor

Dimension row-count validation:
dim_account: source=47, warehouse=47 — PASS
dim_client: source=28, warehouse=28 — PASS
dim_date: source=2,557, warehouse=2,557 — PASS
dim_department: source=11, warehouse=11 — PASS
dim_employee: source=1,606, warehouse=1,606 — PASS
dim_leave_type: source=6, warehouse=6 — PASS
dim_location: source=9, warehouse=9 — PASS
dim_role: source=58, warehouse=58 — PASS
dim_sprint: source=38, warehouse=38 — PASS
dim_team: source=16, warehouse=16 — PASS
dim_vendor: source=30, warehouse=30 — PASS


In [6]:
# Fact tables to copy into the warehouse
fact_tables = [
    "fact_anomaly",
    "fact_attendance_monthly",
    "fact_attrition_risk",
    "fact_expense",
    "fact_forecast",
    "fact_headcount_monthly",
    "fact_invoice",
    "fact_invoice_payment",
    "fact_leave",
    "fact_ledger",
    "fact_payroll",
    "fact_task",
]

for table in fact_tables:
    con_wh.execute(f"""
        CREATE TABLE warehouse.{table} AS
        SELECT *
        FROM source_db.raw.{table}
    """)
    print(f"Created warehouse.{table}")

print("\nFact row-count validation:")

fact_results = []

for table in fact_tables:
    source_count = con_wh.execute(
        f"SELECT COUNT(*) FROM source_db.raw.{table}"
    ).fetchone()[0]

    warehouse_count = con_wh.execute(
        f"SELECT COUNT(*) FROM warehouse.{table}"
    ).fetchone()[0]

    fact_results.append({
        "fact_table": table,
        "source_rows": source_count,
        "warehouse_rows": warehouse_count,
        "status": "PASS" if source_count == warehouse_count else "FAIL"
    })

fact_count_report = pd.DataFrame(fact_results)
display(fact_count_report)

assert (fact_count_report["status"] == "PASS").all(), (
    "At least one fact table has a row-count mismatch."
)

print("All fact table row counts match their sources.")

Created warehouse.fact_anomaly
Created warehouse.fact_attendance_monthly
Created warehouse.fact_attrition_risk
Created warehouse.fact_expense
Created warehouse.fact_forecast
Created warehouse.fact_headcount_monthly
Created warehouse.fact_invoice
Created warehouse.fact_invoice_payment
Created warehouse.fact_leave
Created warehouse.fact_ledger
Created warehouse.fact_payroll
Created warehouse.fact_task

Fact row-count validation:


,fact_table,source_rows,warehouse_rows,status
0,fact_anomaly,405,405,PASS
1,fact_attendance_monthly,7343,7343,PASS
2,fact_attrition_risk,430,430,PASS
3,fact_expense,501,501,PASS
4,fact_forecast,72,72,PASS
5,fact_headcount_monthly,891,891,PASS
6,fact_invoice,743,743,PASS
7,fact_invoice_payment,709,709,PASS
8,fact_leave,3441,3441,PASS
9,fact_ledger,6784,6784,PASS


All fact table row counts match their sources.


In [7]:
# Consolidated source-to-warehouse row-count reconciliation

warehouse_tables = dimension_tables + fact_tables

reconciliation_results = []

for table in warehouse_tables:
    source_count = con_wh.execute(
        f"SELECT COUNT(*) FROM source_db.raw.{table}"
    ).fetchone()[0]

    warehouse_count = con_wh.execute(
        f"SELECT COUNT(*) FROM warehouse.{table}"
    ).fetchone()[0]

    reconciliation_results.append({
        "table_name": table,
        "source_rows": source_count,
        "warehouse_rows": warehouse_count,
        "difference": warehouse_count - source_count,
        "status": (
            "PASS" if source_count == warehouse_count else "FAIL"
        )
    })

reconciliation_df = pd.DataFrame(reconciliation_results)

display(reconciliation_df)

print("\nTables checked:", len(reconciliation_df))
print("Tables passed:", (reconciliation_df["status"] == "PASS").sum())
print("Tables failed:", (reconciliation_df["status"] == "FAIL").sum())
print("Total source rows:", reconciliation_df["source_rows"].sum())
print("Total warehouse rows:", reconciliation_df["warehouse_rows"].sum())

,table_name,source_rows,warehouse_rows,difference,status
0,dim_account,47,47,0,PASS
1,dim_client,28,28,0,PASS
2,dim_date,2557,2557,0,PASS
3,dim_department,11,11,0,PASS
4,dim_employee,1606,1606,0,PASS
5,dim_leave_type,6,6,0,PASS
6,dim_location,9,9,0,PASS
7,dim_role,58,58,0,PASS
8,dim_sprint,38,38,0,PASS
9,dim_team,16,16,0,PASS



Tables checked: 23
Tables passed: 23
Tables failed: 0
Total source rows: 61937
Total warehouse rows: 61937


In [8]:
# Validate employee foreign keys in warehouse fact tables

employee_fk_checks = {
    "fact_payroll": "employee_key",
    "fact_attendance_monthly": "employee_key",
    "fact_attrition_risk": "employee_key",
    "fact_leave": "employee_key",
}

employee_fk_results = []

for table, key in employee_fk_checks.items():
    result = con_wh.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE f."{key}" IS NULL
            ) AS null_keys,
            COUNT(*) FILTER (
                WHERE f."{key}" IS NOT NULL
                  AND d.employee_key IS NULL
            ) AS orphan_keys
        FROM warehouse."{table}" f
        LEFT JOIN warehouse.dim_employee d
            ON f."{key}" = d.employee_key
    """).fetchone()

    employee_fk_results.append({
        "fact_table": table,
        "total_rows": result[0],
        "null_keys": result[1],
        "orphan_keys": result[2],
        "status": (
            "PASS"
            if result[1] == 0 and result[2] == 0
            else "REVIEW"
        )
    })

employee_fk_report = pd.DataFrame(employee_fk_results)
display(employee_fk_report)

,fact_table,total_rows,null_keys,orphan_keys,status
0,fact_payroll,7344,0,2,REVIEW
1,fact_attendance_monthly,7343,0,0,PASS
2,fact_attrition_risk,430,0,0,PASS
3,fact_leave,3441,0,0,PASS


In [9]:
# Validate date and department relationships

relationship_checks = [
    ("fact_payroll", "date_key", "dim_date", "date_key"),
    ("fact_payroll", "department_id", "dim_department", "department_id"),
    ("fact_attendance_monthly", "date_key", "dim_date", "date_key"),
    ("fact_attendance_monthly", "department_id", "dim_department", "department_id"),
    ("fact_attrition_risk", "as_of_date_key", "dim_date", "date_key"),
    ("fact_attrition_risk", "department_id", "dim_department", "department_id"),
    ("fact_headcount_monthly", "date_key", "dim_date", "date_key"),
    ("fact_headcount_monthly", "department_id", "dim_department", "department_id"),
    ("fact_ledger", "date_key", "dim_date", "date_key"),
    ("fact_forecast", "department_id", "dim_department", "department_id"),
]

results = []

for fact, fk, dim, pk in relationship_checks:
    orphan_count = con_wh.execute(f"""
        SELECT COUNT(*)
        FROM warehouse."{fact}" f
        LEFT JOIN warehouse."{dim}" d
            ON f."{fk}" = d."{pk}"
        WHERE f."{fk}" IS NOT NULL
          AND d."{pk}" IS NULL
    """).fetchone()[0]

    results.append({
        "relationship": f"{fact}.{fk} -> {dim}.{pk}",
        "orphan_rows": orphan_count,
        "status": "PASS" if orphan_count == 0 else "REVIEW"
    })

relationship_report = pd.DataFrame(results)
display(relationship_report)

,relationship,orphan_rows,status
0,fact_payroll.date_key -> dim_date.date_key,0,PASS
1,fact_payroll.department_id -> dim_department.d...,0,PASS
2,fact_attendance_monthly.date_key -> dim_date.d...,0,PASS
3,fact_attendance_monthly.department_id -> dim_d...,0,PASS
4,fact_attrition_risk.as_of_date_key -> dim_date...,0,PASS
5,fact_attrition_risk.department_id -> dim_depar...,0,PASS
6,fact_headcount_monthly.date_key -> dim_date.da...,0,PASS
7,fact_headcount_monthly.department_id -> dim_de...,0,PASS
8,fact_ledger.date_key -> dim_date.date_key,0,PASS
9,fact_forecast.department_id -> dim_department....,12,REVIEW


In [10]:
# Inspect forecast rows with unmatched departments
forecast_orphans = con_wh.execute("""
    SELECT
        f.forecast_id,
        f.metric,
        f.department_id,
        f.forecast_month,
        f.forecast_month_key,
        f.forecast_value
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.dim_department d
        ON f.department_id = d.department_id
    WHERE f.department_id IS NOT NULL
      AND d.department_id IS NULL
    ORDER BY f.department_id, f.forecast_month
""").fetchdf()

print("Forecast rows with unmatched departments:", len(forecast_orphans))
display(forecast_orphans)

Forecast rows with unmatched departments: 12


,forecast_id,metric,department_id,forecast_month,forecast_month_key,forecast_value
0,FC-0001,headcount_end,ALL,2026-10-01,20261031,426.30
1,FC-0067,attrition_rate_monthly_pct,ALL,2026-10-01,20261031,1.02
2,FC-0002,headcount_end,ALL,2026-11-01,20261130,429.70
3,FC-0068,attrition_rate_monthly_pct,ALL,2026-11-01,20261130,1.01
4,FC-0003,headcount_end,ALL,2026-12-01,20261231,431.40
5,FC-0069,attrition_rate_monthly_pct,ALL,2026-12-01,20261231,1.66
6,FC-0004,headcount_end,ALL,2027-01-01,20270131,436.00
7,FC-0070,attrition_rate_monthly_pct,ALL,2027-01-01,20270131,0.92
8,FC-0005,headcount_end,ALL,2027-02-01,20270228,439.70
9,FC-0071,attrition_rate_monthly_pct,ALL,2027-02-01,20270228,1.23


In [11]:
# Check forecast month keys against the date dimension
forecast_date_check = con_wh.execute("""
    SELECT
        f.forecast_id,
        f.forecast_month,
        f.forecast_month_key,
        d.date_key AS matched_date_key,
        CASE
            WHEN d.date_key IS NULL THEN 'MISSING DATE KEY'
            ELSE 'MATCHED'
        END AS date_status
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.dim_date d
        ON f.forecast_month_key = d.date_key
    ORDER BY f.forecast_month
""").fetchdf()

display(forecast_date_check)

print("\nDate-key validation summary:")
display(
    forecast_date_check["date_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)

,forecast_id,forecast_month,forecast_month_key,matched_date_key,date_status
0,FC-0067,2026-10-01,20261031,20261031,MATCHED
1,FC-0043,2026-10-01,20261031,20261031,MATCHED
2,FC-0001,2026-10-01,20261031,20261031,MATCHED
3,FC-0061,2026-10-01,20261031,20261031,MATCHED
4,FC-0007,2026-10-01,20261031,20261031,MATCHED
...,...,...,...,...,...
67,FC-0018,2027-03-01,20270331,<NA>,MISSING DATE KEY
68,FC-0066,2027-03-01,20270331,<NA>,MISSING DATE KEY
69,FC-0012,2027-03-01,20270331,<NA>,MISSING DATE KEY
70,FC-0072,2027-03-01,20270331,<NA>,MISSING DATE KEY



Date-key validation summary:


,status,rows
0,MATCHED,36
1,MISSING DATE KEY,36


In [12]:
# Summarize forecast date-key coverage by month
forecast_date_summary = con_wh.execute("""
    SELECT
        f.forecast_month,
        f.forecast_month_key,
        COUNT(*) AS forecast_rows,
        COUNT(d.date_key) AS matched_date_rows,
        COUNT(*) - COUNT(d.date_key) AS missing_date_rows
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.dim_date d
        ON f.forecast_month_key = d.date_key
    GROUP BY
        f.forecast_month,
        f.forecast_month_key
    ORDER BY f.forecast_month
""").fetchdf()

display(forecast_date_summary)

print("\nDate dimension range:")
display(con_wh.execute("""
    SELECT
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,
        MIN(date_key) AS minimum_date_key,
        MAX(date_key) AS maximum_date_key
    FROM warehouse.dim_date
""").fetchdf())

,forecast_month,forecast_month_key,forecast_rows,matched_date_rows,missing_date_rows
0,2026-10-01,20261031,12,12,0
1,2026-11-01,20261130,12,12,0
2,2026-12-01,20261231,12,12,0
3,2027-01-01,20270131,12,0,12
4,2027-02-01,20270228,12,0,12
5,2027-03-01,20270331,12,0,12



Date dimension range:


,earliest_date,latest_date,minimum_date_key,maximum_date_key
0,2020-01-01,2026-12-31,20200101,20261231


In [13]:
# Inspect warehouse date dimension structure
date_columns = con_wh.execute("""
    DESCRIBE warehouse.dim_date
""").fetchdf()

display(date_columns)

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,date_key,BIGINT,YES,None,None,None
2,year,BIGINT,YES,None,None,None
3,quarter,BIGINT,YES,None,None,None
4,month_number,BIGINT,YES,None,None,None
5,month_name,VARCHAR,YES,None,None,None
6,year_month,VARCHAR,YES,None,None,None
7,week_of_year,BIGINT,YES,None,None,None
8,day_of_week,VARCHAR,YES,None,None,None
9,is_weekend,VARCHAR,YES,None,None,None


In [14]:
# Inspect date attributes around the end of the source calendar
display(con_wh.execute("""
    SELECT *
    FROM warehouse.dim_date
    WHERE date BETWEEN DATE '2026-12-25' AND DATE '2026-12-31'
    ORDER BY date
""").fetchdf())

,date,date_key,year,quarter,month_number,month_name,year_month,week_of_year,day_of_week,is_weekend,is_holiday,is_working_day,financial_year,fiscal_quarter,fiscal_month_number,is_month_end,month_start_date,month_end_date
0,2026-12-25,20261225,2026,4,12,December,2026-12,52,Friday,N,N,Y,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
1,2026-12-26,20261226,2026,4,12,December,2026-12,52,Saturday,Y,N,N,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
2,2026-12-27,20261227,2026,4,12,December,2026-12,52,Sunday,Y,N,N,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
3,2026-12-28,20261228,2026,4,12,December,2026-12,53,Monday,N,N,Y,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
4,2026-12-29,20261229,2026,4,12,December,2026-12,53,Tuesday,N,N,Y,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
5,2026-12-30,20261230,2026,4,12,December,2026-12,53,Wednesday,N,N,Y,FY2026-27,Q3,9,N,2026-12-01,2026-12-31
6,2026-12-31,20261231,2026,4,12,December,2026-12,53,Thursday,N,N,Y,FY2026-27,Q3,9,Y,2026-12-01,2026-12-31


In [15]:
# Inspect existing fiscal-month and date-flag conventions
display(con_wh.execute("""
    SELECT
        date,
        day_of_week,
        is_weekend,
        is_holiday,
        is_working_day,
        financial_year,
        fiscal_quarter,
        fiscal_month_number,
        is_month_end
    FROM warehouse.dim_date
    WHERE date BETWEEN DATE '2026-04-01' AND DATE '2026-04-07'
       OR date BETWEEN DATE '2026-10-01' AND DATE '2026-10-07'
    ORDER BY date
""").fetchdf())

,date,day_of_week,is_weekend,is_holiday,is_working_day,financial_year,fiscal_quarter,fiscal_month_number,is_month_end
0,2026-04-01,Wednesday,N,N,Y,FY2026-27,Q1,1,N
1,2026-04-02,Thursday,N,N,Y,FY2026-27,Q1,1,N
2,2026-04-03,Friday,N,Y,N,FY2026-27,Q1,1,N
3,2026-04-04,Saturday,Y,N,N,FY2026-27,Q1,1,N
4,2026-04-05,Sunday,Y,N,N,FY2026-27,Q1,1,N
5,2026-04-06,Monday,N,N,Y,FY2026-27,Q1,1,N
6,2026-04-07,Tuesday,N,N,Y,FY2026-27,Q1,1,N
7,2026-10-01,Thursday,N,N,Y,FY2026-27,Q3,7,N
8,2026-10-02,Friday,N,N,Y,FY2026-27,Q3,7,N
9,2026-10-03,Saturday,Y,N,N,FY2026-27,Q3,7,N


In [16]:
from datetime import date, timedelta
import pandas as pd

# Generate dates beyond the existing calendar's end
start_date = date(2027, 1, 1)
end_date = date(2027, 3, 31)

new_dates = []
current_date = start_date

while current_date <= end_date:
    month_start = current_date.replace(day=1)

    if current_date.month == 12:
        month_end = current_date.replace(day=31)
    else:
        next_month = current_date.replace(
            year=current_date.year + (current_date.month // 12),
            month=(current_date.month % 12) + 1,
            day=1
        )
        month_end = next_month - timedelta(days=1)

    month = current_date.month
    fiscal_month = ((month - 4) % 12) + 1
    fiscal_quarter = f"Q{((fiscal_month - 1) // 3) + 1}"
    financial_year = (
        f"FY{current_date.year}-{str(current_date.year + 1)[-2:]}"
        if month >= 4
        else f"FY{current_date.year - 1}-{str(current_date.year)[-2:]}"
    )

    new_dates.append({
        "date": current_date,
        "date_key": int(current_date.strftime("%Y%m%d")),
        "year": current_date.year,
        "quarter": (month - 1) // 3 + 1,
        "month_number": month,
        "month_name": current_date.strftime("%B"),
        "year_month": current_date.strftime("%Y-%m"),
        "week_of_year": current_date.isocalendar().week,
        "day_of_week": current_date.strftime("%A"),
        "is_weekend": "Y" if current_date.weekday() >= 5 else "N",
        "is_holiday": None,
        "is_working_day": None,
        "financial_year": financial_year,
        "fiscal_quarter": fiscal_quarter,
        "fiscal_month_number": fiscal_month,
        "is_month_end": "Y" if current_date == month_end else "N",
        "month_start_date": month_start,
        "month_end_date": month_end,
    })

    current_date += timedelta(days=1)

calendar_extension = pd.DataFrame(new_dates)

print("Dates prepared:", len(calendar_extension))
print("First date:", calendar_extension["date"].min())
print("Last date:", calendar_extension["date"].max())
display(calendar_extension.head())
display(calendar_extension.tail())

Dates prepared: 90
First date: 2027-01-01
Last date: 2027-03-31


,date,date_key,year,quarter,month_number,month_name,year_month,week_of_year,day_of_week,is_weekend,is_holiday,is_working_day,financial_year,fiscal_quarter,fiscal_month_number,is_month_end,month_start_date,month_end_date
0,2027-01-01,20270101,2027,1,1,January,2027-01,53,Friday,N,None,None,FY2026-27,Q4,10,N,2027-01-01,2027-01-31
1,2027-01-02,20270102,2027,1,1,January,2027-01,53,Saturday,Y,None,None,FY2026-27,Q4,10,N,2027-01-01,2027-01-31
2,2027-01-03,20270103,2027,1,1,January,2027-01,53,Sunday,Y,None,None,FY2026-27,Q4,10,N,2027-01-01,2027-01-31
3,2027-01-04,20270104,2027,1,1,January,2027-01,1,Monday,N,None,None,FY2026-27,Q4,10,N,2027-01-01,2027-01-31
4,2027-01-05,20270105,2027,1,1,January,2027-01,1,Tuesday,N,None,None,FY2026-27,Q4,10,N,2027-01-01,2027-01-31


,date,date_key,year,quarter,month_number,month_name,year_month,week_of_year,day_of_week,is_weekend,is_holiday,is_working_day,financial_year,fiscal_quarter,fiscal_month_number,is_month_end,month_start_date,month_end_date
85,2027-03-27,20270327,2027,1,3,March,2027-03,12,Saturday,Y,None,None,FY2026-27,Q4,12,N,2027-03-01,2027-03-31
86,2027-03-28,20270328,2027,1,3,March,2027-03,12,Sunday,Y,None,None,FY2026-27,Q4,12,N,2027-03-01,2027-03-31
87,2027-03-29,20270329,2027,1,3,March,2027-03,13,Monday,N,None,None,FY2026-27,Q4,12,N,2027-03-01,2027-03-31
88,2027-03-30,20270330,2027,1,3,March,2027-03,13,Tuesday,N,None,None,FY2026-27,Q4,12,N,2027-03-01,2027-03-31
89,2027-03-31,20270331,2027,1,3,March,2027-03,13,Wednesday,N,None,None,FY2026-27,Q4,12,Y,2027-03-01,2027-03-31


In [17]:
# Validate the proposed calendar extension

assert len(calendar_extension) == 90
assert calendar_extension["date"].is_unique
assert calendar_extension["date_key"].is_unique

assert calendar_extension["date"].min() == date(2027, 1, 1)
assert calendar_extension["date"].max() == date(2027, 3, 31)

# Verify every day is consecutive
date_gaps = (
    calendar_extension["date"].diff().dropna()
    .ne(pd.Timedelta(days=1))
    .sum()
)

# Check fiscal month and quarter labels
fiscal_labels = (
    calendar_extension[
        ["month_name", "financial_year", "fiscal_quarter",
         "fiscal_month_number"]
    ]
    .drop_duplicates()
    .sort_values("fiscal_month_number")
)

# Check month-end flags
month_ends = calendar_extension.loc[
    calendar_extension["is_month_end"] == "Y",
    ["date", "month_end_date"]
]

print("Total dates:", len(calendar_extension))
print("Duplicate dates:", calendar_extension["date"].duplicated().sum())
print("Duplicate date keys:", calendar_extension["date_key"].duplicated().sum())
print("Date gaps:", date_gaps)
print("\nFiscal labels:")
display(fiscal_labels)
print("\nMonth-end validation:")
display(month_ends)

Total dates: 90
Duplicate dates: 0
Duplicate date keys: 0
Date gaps: 0

Fiscal labels:


,month_name,financial_year,fiscal_quarter,fiscal_month_number
0,January,FY2026-27,Q4,10
31,February,FY2026-27,Q4,11
59,March,FY2026-27,Q4,12



Month-end validation:


,date,month_end_date
30,2027-01-31,2027-01-31
58,2027-02-28,2027-02-28
89,2027-03-31,2027-03-31


In [18]:
# Check for existing date keys before inserting
existing_keys = con_wh.execute("""
    SELECT COUNT(*)
    FROM warehouse.dim_date
    WHERE date_key BETWEEN 20270101 AND 20270331
""").fetchone()[0]

print("Existing 2027 date keys:", existing_keys)

assert existing_keys == 0, (
    "Some 2027 dates already exist. Stop and investigate before inserting."
)

# Register the validated calendar extension
con_wh.register("calendar_extension_df", calendar_extension)

# Insert only the missing future dates into the warehouse
con_wh.execute("""
    INSERT INTO warehouse.dim_date
    SELECT *
    FROM calendar_extension_df
""")

con_wh.unregister("calendar_extension_df")

print("Calendar extension inserted successfully!")

Existing 2027 date keys: 0
Calendar extension inserted successfully!


In [19]:
# 1. Verify the warehouse calendar
warehouse_calendar = con_wh.execute("""
    SELECT
        COUNT(*) AS total_dates,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(DISTINCT date_key) AS unique_date_keys
    FROM warehouse.dim_date
""").df()

print("WAREHOUSE CALENDAR")
display(warehouse_calendar)

# 2. Confirm the original source calendar remains unchanged
source_calendar = con_wh.execute("""
    SELECT
        COUNT(*) AS total_dates,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM source_db.raw.dim_date
""").df()

print("ORIGINAL SOURCE CALENDAR")
display(source_calendar)

# 3. Inspect forecast columns before validating the join
print("FORECAST TABLE COLUMNS")
display(con_wh.execute("""
    DESCRIBE warehouse.fact_forecast
""").df())

WAREHOUSE CALENDAR


,total_dates,first_date,last_date,unique_date_keys
0,2647,2020-01-01,2027-03-31,2647


ORIGINAL SOURCE CALENDAR


,total_dates,first_date,last_date
0,2557,2020-01-01,2026-12-31


FORECAST TABLE COLUMNS


,column_name,column_type,null,key,default,extra
0,forecast_id,VARCHAR,YES,None,None,None
1,metric,VARCHAR,YES,None,None,None
2,department_id,VARCHAR,YES,None,None,None
3,forecast_month,DATE,YES,None,None,None
4,forecast_value,DOUBLE,YES,None,None,None
5,lower_80,DOUBLE,YES,None,None,None
6,upper_80,DOUBLE,YES,None,None,None
7,method,VARCHAR,YES,None,None,None
8,trained_through,VARCHAR,YES,None,None,None
9,model_version,VARCHAR,YES,None,None,None


In [20]:
# Check whether every forecast month maps to the warehouse calendar
forecast_date_check = con_wh.execute("""
    SELECT
        COUNT(*) AS total_forecast_records,
        COUNT(d.date_key) AS matched_date_keys,
        COUNT(*) - COUNT(d.date_key) AS unmatched_date_keys
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.dim_date d
        ON f.forecast_month_key = d.date_key
""").df()

print("FORECAST DATE-KEY VALIDATION")
display(forecast_date_check)

# Show any remaining unmatched forecast records
unmatched_forecasts = con_wh.execute("""
    SELECT
        f.forecast_id,
        f.forecast_month,
        f.forecast_month_key,
        f.department_id,
        f.metric
    FROM warehouse.fact_forecast f
    LEFT JOIN warehouse.dim_date d
        ON f.forecast_month_key = d.date_key
    WHERE d.date_key IS NULL
    ORDER BY f.forecast_month, f.department_id
""").df()

print("UNMATCHED FORECAST RECORDS")
display(unmatched_forecasts)

# Confirm the expected validation result
total = forecast_date_check.loc[0, "total_forecast_records"]
matched = forecast_date_check.loc[0, "matched_date_keys"]
unmatched = forecast_date_check.loc[0, "unmatched_date_keys"]

assert total == 72, f"Expected 72 forecast records, found {total}"
assert unmatched == 0, f"Found {unmatched} unmatched forecast date keys"

print("PASS: All forecast date keys match the warehouse calendar.")

FORECAST DATE-KEY VALIDATION


,total_forecast_records,matched_date_keys,unmatched_date_keys
0,72,72,0


UNMATCHED FORECAST RECORDS


,forecast_id,forecast_month,forecast_month_key,department_id,metric


PASS: All forecast date keys match the warehouse calendar.


In [21]:
tables_to_inspect = [
    "dim_account",
    "dim_client",
    "dim_department",
    "dim_employee",
    "dim_leave_type",
    "dim_location",
    "dim_role",
    "dim_sprint",
    "dim_team",
    "dim_vendor",
    "fact_expense",
    "fact_invoice",
    "fact_invoice_payment",
    "fact_leave",
    "fact_task",
    "fact_anomaly",
    "fact_headcount_monthly",
]

for table_name in tables_to_inspect:
    print(f"\n--- {table_name} ---")
    columns = con_wh.execute(
        f"DESCRIBE warehouse.{table_name}"
    ).df()
    print(columns["column_name"].tolist())


--- dim_account ---
['account_code', 'account_name', 'account_type', 'normal_balance', 'parent_code', 'is_group', 'statement', 'is_active']

--- dim_client ---
['client_id', 'client_name', 'industry', 'city', 'state_or_region', 'country', 'client_type', 'gst_registration', 'payment_terms_days', 'payment_behaviour', 'primary_service_line', 'contact_email', 'onboarded_on', 'is_active']

--- dim_department ---
['department_id', 'department_name', 'cost_center', 'description', 'is_leadership']

--- dim_employee ---
['employee_key', 'employee_id', 'full_name', 'gender', 'date_of_birth', 'hire_date', 'exit_date', 'employment_status', 'employment_type', 'department_id', 'role_id', 'role_title', 'job_level', 'manager_id', 'work_location', 'work_mode', 'highest_education', 'valid_from', 'valid_to', 'is_current', 'change_reason']

--- dim_leave_type ---
['leave_type_code', 'leave_type_name', 'pay_type', 'annual_quota_days', 'prorated_for_joiners', 'carry_forward_allowed', 'max_carry_forward_day

In [22]:
# PQ-02: Foreign-key validation for remaining warehouse tables
# Read-only checks — no data will be modified.

relationship_checks = {
    "Expense -> Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_expense f
        LEFT JOIN warehouse.dim_date d
            ON f.expense_date_key = d.date_key
        WHERE f.expense_date_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Expense -> Account": """
        SELECT COUNT(*)
        FROM warehouse.fact_expense f
        LEFT JOIN warehouse.dim_account d
            ON f.category_account = d.account_code
        WHERE f.category_account IS NOT NULL
          AND d.account_code IS NULL
    """,

    "Expense -> Vendor": """
        SELECT COUNT(*)
        FROM warehouse.fact_expense f
        LEFT JOIN warehouse.dim_vendor d
            ON f.vendor_id = d.vendor_id
        WHERE f.vendor_id IS NOT NULL
          AND d.vendor_id IS NULL
    """,

    "Expense -> Department": """
        SELECT COUNT(*)
        FROM warehouse.fact_expense f
        LEFT JOIN warehouse.dim_department d
            ON f.department_id = d.department_id
        WHERE f.department_id IS NOT NULL
          AND d.department_id IS NULL
    """,

    "Invoice -> Client": """
        SELECT COUNT(*)
        FROM warehouse.fact_invoice f
        LEFT JOIN warehouse.dim_client d
            ON f.client_id = d.client_id
        WHERE f.client_id IS NOT NULL
          AND d.client_id IS NULL
    """,

    "Invoice -> Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_invoice f
        LEFT JOIN warehouse.dim_date d
            ON f.invoice_date_key = d.date_key
        WHERE f.invoice_date_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Invoice -> Account": """
        SELECT COUNT(*)
        FROM warehouse.fact_invoice f
        LEFT JOIN warehouse.dim_account d
            ON f.service_line_account = d.account_code
        WHERE f.service_line_account IS NOT NULL
          AND d.account_code IS NULL
    """,

    "Payment -> Invoice": """
        SELECT COUNT(*)
        FROM warehouse.fact_invoice_payment f
        LEFT JOIN warehouse.fact_invoice i
            ON f.invoice_id = i.invoice_id
        WHERE f.invoice_id IS NOT NULL
          AND i.invoice_id IS NULL
    """,

    "Payment -> Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_invoice_payment f
        LEFT JOIN warehouse.dim_date d
            ON f.payment_date_key = d.date_key
        WHERE f.payment_date_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Leave -> Start Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_leave f
        LEFT JOIN warehouse.dim_date d
            ON f.start_date_key = d.date_key
        WHERE f.start_date_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Leave -> End Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_leave f
        LEFT JOIN warehouse.dim_date d
            ON f.end_date_key = d.date_key
        WHERE f.end_date_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Leave -> Leave Type": """
        SELECT COUNT(*)
        FROM warehouse.fact_leave f
        LEFT JOIN warehouse.dim_leave_type d
            ON f.leave_type_code = d.leave_type_code
        WHERE f.leave_type_code IS NOT NULL
          AND d.leave_type_code IS NULL
    """,

    "Leave -> Employee Key": """
        SELECT COUNT(*)
        FROM warehouse.fact_leave f
        LEFT JOIN warehouse.dim_employee d
            ON f.employee_key = d.employee_key
        WHERE f.employee_key IS NOT NULL
          AND d.employee_key IS NULL
    """,

    "Task -> Team": """
        SELECT COUNT(*)
        FROM warehouse.fact_task f
        LEFT JOIN warehouse.dim_team d
            ON f.team_id = d.team_id
        WHERE f.team_id IS NOT NULL
          AND d.team_id IS NULL
    """,

    "Task -> Sprint": """
        SELECT COUNT(*)
        FROM warehouse.fact_task f
        LEFT JOIN warehouse.dim_sprint d
            ON f.sprint_id = d.sprint_id
        WHERE f.sprint_id IS NOT NULL
          AND d.sprint_id IS NULL
    """,

    "Task -> Assignee": """
        SELECT COUNT(*)
        FROM warehouse.fact_task f
        LEFT JOIN (
            SELECT DISTINCT employee_id
            FROM warehouse.dim_employee
            WHERE is_current = 'Y'
        ) d
            ON f.assignee_employee_id = d.employee_id
        WHERE f.assignee_employee_id IS NOT NULL
          AND d.employee_id IS NULL
    """,

    "Anomaly -> Employee": """
        SELECT COUNT(*)
        FROM warehouse.fact_anomaly f
        LEFT JOIN (
            SELECT DISTINCT employee_id
            FROM warehouse.dim_employee
            WHERE is_current = 'Y'
        ) d
            ON f.employee_id = d.employee_id
        WHERE f.employee_id IS NOT NULL
          AND d.employee_id IS NULL
    """,

    "Anomaly -> Period Date": """
        SELECT COUNT(*)
        FROM warehouse.fact_anomaly f
        LEFT JOIN warehouse.dim_date d
            ON f.period_key = d.date_key
        WHERE f.period_key IS NOT NULL
          AND d.date_key IS NULL
    """,

    "Role -> Department": """
        SELECT COUNT(*)
        FROM warehouse.dim_role f
        LEFT JOIN warehouse.dim_department d
            ON f.department_id = d.department_id
        WHERE f.department_id IS NOT NULL
          AND d.department_id IS NULL
    """,

    "Team -> Department": """
        SELECT COUNT(*)
        FROM warehouse.dim_team f
        LEFT JOIN warehouse.dim_department d
            ON f.department_id = d.department_id
        WHERE f.department_id IS NOT NULL
          AND d.department_id IS NULL
    """,

    "Employee -> Department": """
        SELECT COUNT(*)
        FROM warehouse.dim_employee f
        LEFT JOIN warehouse.dim_department d
            ON f.department_id = d.department_id
        WHERE f.department_id IS NOT NULL
          AND d.department_id IS NULL
    """,

    "Employee -> Role": """
        SELECT COUNT(*)
        FROM warehouse.dim_employee f
        LEFT JOIN warehouse.dim_role d
            ON f.role_id = d.role_id
        WHERE f.role_id IS NOT NULL
          AND d.role_id IS NULL
    """,

    "Employee -> Location": """
        SELECT COUNT(*)
        FROM warehouse.dim_employee f
        LEFT JOIN warehouse.dim_location d
            ON f.work_location = d.work_location
        WHERE f.work_location IS NOT NULL
          AND d.work_location IS NULL
    """,

    "Vendor -> Account": """
        SELECT COUNT(*)
        FROM warehouse.dim_vendor f
        LEFT JOIN warehouse.dim_account d
            ON f.primary_expense_account = d.account_code
        WHERE f.primary_expense_account IS NOT NULL
          AND d.account_code IS NULL
    """,

    "Account -> Parent Account": """
        SELECT COUNT(*)
        FROM warehouse.dim_account f
        LEFT JOIN warehouse.dim_account d
            ON f.parent_code = d.account_code
        WHERE f.parent_code IS NOT NULL
          AND d.account_code IS NULL
    """
}

relationship_results = []

for relationship, query in relationship_checks.items():
    unmatched_count = con_wh.execute(query).fetchone()[0]

    relationship_results.append({
        "relationship": relationship,
        "unmatched_records": unmatched_count,
        "status": "PASS" if unmatched_count == 0 else "REVIEW"
    })

relationship_df = pd.DataFrame(relationship_results)

print("RELATIONSHIP VALIDATION SUMMARY")
display(relationship_df)

print("Relationships checked:", len(relationship_df))
print("Passed:", (relationship_df["status"] == "PASS").sum())
print("Need review:", (relationship_df["status"] == "REVIEW").sum())

RELATIONSHIP VALIDATION SUMMARY


,relationship,unmatched_records,status
0,Expense -> Date,0,PASS
1,Expense -> Account,0,PASS
2,Expense -> Vendor,0,PASS
3,Expense -> Department,0,PASS
4,Invoice -> Client,0,PASS
5,Invoice -> Date,0,PASS
6,Invoice -> Account,0,PASS
7,Payment -> Invoice,0,PASS
8,Payment -> Date,0,PASS
9,Leave -> Start Date,0,PASS


Relationships checked: 25
Passed: 25
Need review: 0


In [23]:
# PQ-02: Primary-key candidate validation
# Read-only — this cell does not modify any tables.

primary_key_candidates = {
    "dim_account": ["account_code"],
    "dim_client": ["client_id"],
    "dim_date": ["date_key"],
    "dim_department": ["department_id"],
    "dim_employee": ["employee_key"],
    "dim_leave_type": ["leave_type_code"],
    "dim_location": ["work_location"],
    "dim_role": ["role_id"],
    "dim_sprint": ["sprint_id"],
    "dim_team": ["team_id"],
    "dim_vendor": ["vendor_id"],
    "fact_anomaly": ["result_id"],
    "fact_attendance_monthly": ["attendance_fact_id"],
    "fact_attrition_risk": ["prediction_id"],
    "fact_expense": ["expense_id"],
    "fact_forecast": ["forecast_id"],
    "fact_headcount_monthly": ["date_key", "department_id"],
    "fact_invoice": ["invoice_id"],
    "fact_invoice_payment": ["payment_id"],
    "fact_leave": ["leave_request_id"],
    "fact_ledger": ["line_id"],
    "fact_payroll": ["payroll_item_id"],
    "fact_task": ["task_id"],
}

pk_validation_results = []

for table_name, key_columns in primary_key_candidates.items():
    key_sql = ", ".join(f'"{col}"' for col in key_columns)

    query = f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) - COUNT({key_sql.split(', ')[0]}) AS null_key_rows,
            COUNT(*) - COUNT(DISTINCT ({key_sql})) AS duplicate_key_rows
        FROM warehouse."{table_name}"
    """

    total_rows, null_key_rows, duplicate_key_rows = (
        con_wh.execute(query).fetchone()
    )

    status = (
        "PASS"
        if null_key_rows == 0 and duplicate_key_rows == 0
        else "REVIEW"
    )

    pk_validation_results.append({
        "table": table_name,
        "candidate_key": " + ".join(key_columns),
        "total_rows": total_rows,
        "null_key_rows": null_key_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "status": status
    })

pk_validation_df = pd.DataFrame(pk_validation_results)

print("PRIMARY-KEY CANDIDATE VALIDATION")
display(pk_validation_df)

print("Tables checked:", len(pk_validation_df))
print("Passed:", (pk_validation_df["status"] == "PASS").sum())
print("Need review:", (pk_validation_df["status"] == "REVIEW").sum())

PRIMARY-KEY CANDIDATE VALIDATION


,table,candidate_key,total_rows,null_key_rows,duplicate_key_rows,status
0,dim_account,account_code,47,0,0,PASS
1,dim_client,client_id,28,0,0,PASS
2,dim_date,date_key,2647,0,0,PASS
3,dim_department,department_id,11,0,0,PASS
4,dim_employee,employee_key,1606,0,0,PASS
5,dim_leave_type,leave_type_code,6,0,0,PASS
6,dim_location,work_location,9,0,0,PASS
7,dim_role,role_id,58,0,0,PASS
8,dim_sprint,sprint_id,38,0,0,PASS
9,dim_team,team_id,16,0,0,PASS


Tables checked: 23
Passed: 23
Need review: 0


In [24]:
# Test primary-key support safely on a temporary table

con_wh.execute("""
    CREATE TEMP TABLE pq02_pk_test (
        test_id INTEGER
    )
""")

try:
    con_wh.execute("""
        ALTER TABLE pq02_pk_test
        ADD PRIMARY KEY (test_id)
    """)

    print("PASS: ALTER TABLE ADD PRIMARY KEY is supported.")

except Exception as e:
    print("This DuckDB version does not support that operation directly.")
    print("Details:", str(e))

finally:
    con_wh.execute("DROP TABLE IF EXISTS pq02_pk_test")
    print("Temporary test table removed.")

PASS: ALTER TABLE ADD PRIMARY KEY is supported.
Temporary test table removed.


In [25]:
# PQ-02: Enforce primary keys on all 23 warehouse tables

primary_key_candidates = {
    "dim_account": ["account_code"],
    "dim_client": ["client_id"],
    "dim_date": ["date_key"],
    "dim_department": ["department_id"],
    "dim_employee": ["employee_key"],
    "dim_leave_type": ["leave_type_code"],
    "dim_location": ["work_location"],
    "dim_role": ["role_id"],
    "dim_sprint": ["sprint_id"],
    "dim_team": ["team_id"],
    "dim_vendor": ["vendor_id"],
    "fact_anomaly": ["result_id"],
    "fact_attendance_monthly": ["attendance_fact_id"],
    "fact_attrition_risk": ["prediction_id"],
    "fact_expense": ["expense_id"],
    "fact_forecast": ["forecast_id"],
    "fact_headcount_monthly": ["date_key", "department_id"],
    "fact_invoice": ["invoice_id"],
    "fact_invoice_payment": ["payment_id"],
    "fact_leave": ["leave_request_id"],
    "fact_ledger": ["line_id"],
    "fact_payroll": ["payroll_item_id"],
    "fact_task": ["task_id"],
}

try:
    con_wh.execute("BEGIN TRANSACTION")

    for table_name, key_columns in primary_key_candidates.items():
        columns_sql = ", ".join(f'"{col}"' for col in key_columns)

        sql = f"""
            ALTER TABLE warehouse."{table_name}"
            ADD PRIMARY KEY ({columns_sql})
        """

        con_wh.execute(sql)
        print(f"Primary key added: {table_name} ({', '.join(key_columns)})")

    con_wh.execute("COMMIT")
    print("\nSUCCESS: All 23 primary-key constraints added.")

except Exception as e:
    con_wh.execute("ROLLBACK")
    print("\nFAILED: Changes rolled back.")
    print("Error:", str(e))
    raise

Primary key added: dim_account (account_code)
Primary key added: dim_client (client_id)
Primary key added: dim_date (date_key)
Primary key added: dim_department (department_id)
Primary key added: dim_employee (employee_key)
Primary key added: dim_leave_type (leave_type_code)
Primary key added: dim_location (work_location)
Primary key added: dim_role (role_id)
Primary key added: dim_sprint (sprint_id)
Primary key added: dim_team (team_id)
Primary key added: dim_vendor (vendor_id)
Primary key added: fact_anomaly (result_id)
Primary key added: fact_attendance_monthly (attendance_fact_id)
Primary key added: fact_attrition_risk (prediction_id)
Primary key added: fact_expense (expense_id)
Primary key added: fact_forecast (forecast_id)
Primary key added: fact_headcount_monthly (date_key, department_id)
Primary key added: fact_invoice (invoice_id)
Primary key added: fact_invoice_payment (payment_id)
Primary key added: fact_leave (leave_request_id)
Primary key added: fact_ledger (line_id)
Prima

In [26]:
# Verify primary-key constraints registered in DuckDB

primary_keys_created = con_wh.execute("""
    SELECT
        table_name,
        constraint_column_names
    FROM duckdb_constraints()
    WHERE schema_name = 'warehouse'
      AND constraint_type = 'PRIMARY KEY'
    ORDER BY table_name
""").df()

display(primary_keys_created)

print("Primary-key constraints found:", len(primary_keys_created))

assert len(primary_keys_created) == 23, (
    f"Expected 23 primary keys, found {len(primary_keys_created)}"
)

print("PASS: All 23 primary-key constraints are verified.")

,table_name,constraint_column_names
0,dim_account,[account_code]
1,dim_client,[client_id]
2,dim_date,[date_key]
3,dim_department,[department_id]
4,dim_employee,[employee_key]
5,dim_leave_type,[leave_type_code]
6,dim_location,[work_location]
7,dim_role,[role_id]
8,dim_sprint,[sprint_id]
9,dim_team,[team_id]


Primary-key constraints found: 23
PASS: All 23 primary-key constraints are verified.


In [27]:
# PQ-02: Final warehouse integrity check

# 1. Verify row counts for all 23 warehouse tables
final_row_counts = []

for table_name in list(dimension_tables) + list(fact_tables):
    warehouse_count = con_wh.execute(
        f'SELECT COUNT(*) FROM warehouse."{table_name}"'
    ).fetchone()[0]

    source_count = con_wh.execute(
        f'SELECT COUNT(*) FROM source_db.raw."{table_name}"'
    ).fetchone()[0]

    # dim_date intentionally has 90 additional warehouse-only rows
    expected_warehouse_count = (
        source_count + 90 if table_name == "dim_date" else source_count
    )

    final_row_counts.append({
        "table": table_name,
        "source_rows": source_count,
        "warehouse_rows": warehouse_count,
        "expected_warehouse_rows": expected_warehouse_count,
        "status": (
            "PASS"
            if warehouse_count == expected_warehouse_count
            else "REVIEW"
        )
    })

final_counts_df = pd.DataFrame(final_row_counts)

print("FINAL ROW-COUNT RECONCILIATION")
display(final_counts_df)

print("Tables checked:", len(final_counts_df))
print("Passed:", (final_counts_df["status"] == "PASS").sum())
print("Need review:", (final_counts_df["status"] == "REVIEW").sum())

# 2. Confirm the two known unmatched payroll employee keys
payroll_orphans = con_wh.execute("""
    SELECT payroll_item_id, employee_id, employee_key
    FROM warehouse.fact_payroll p
    WHERE NOT EXISTS (
        SELECT 1
        FROM warehouse.dim_employee e
        WHERE e.employee_key = p.employee_key
    )
""").df()

print("\nUNMATCHED PAYROLL EMPLOYEE RECORDS")
display(payroll_orphans)

# 3. Confirm company-wide forecast rows using department_id = ALL
all_department_forecasts = con_wh.execute("""
    SELECT COUNT(*)
    FROM warehouse.fact_forecast
    WHERE department_id = 'ALL'
""").fetchone()[0]

print("Company-wide forecast records (ALL):", all_department_forecasts)

# 4. Confirm the primary-key constraints remain present
pk_count = con_wh.execute("""
    SELECT COUNT(*)
    FROM duckdb_constraints()
    WHERE schema_name = 'warehouse'
      AND constraint_type = 'PRIMARY KEY'
""").fetchone()[0]

print("Verified primary-key constraints:", pk_count)

assert len(final_counts_df) == 23
assert (final_counts_df["status"] == "PASS").all()
assert len(payroll_orphans) == 2
assert all_department_forecasts == 12
assert pk_count == 23

print("\nPASS: Final warehouse integrity checks completed.")

FINAL ROW-COUNT RECONCILIATION


,table,source_rows,warehouse_rows,expected_warehouse_rows,status
0,dim_account,47,47,47,PASS
1,dim_client,28,28,28,PASS
2,dim_date,2557,2647,2647,PASS
3,dim_department,11,11,11,PASS
4,dim_employee,1606,1606,1606,PASS
5,dim_leave_type,6,6,6,PASS
6,dim_location,9,9,9,PASS
7,dim_role,58,58,58,PASS
8,dim_sprint,38,38,38,PASS
9,dim_team,16,16,16,PASS


Tables checked: 23
Passed: 23
Need review: 0

UNMATCHED PAYROLL EMPLOYEE RECORDS


,payroll_item_id,employee_id,employee_key
0,PI-X0003,EMP-9001,-1
1,PI-X0004,EMP-9002,-1


Company-wide forecast records (ALL): 12
Verified primary-key constraints: 23

PASS: Final warehouse integrity checks completed.


In [28]:
from pathlib import Path

docs_dir = PROJECT_ROOT / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

schema_doc = r"""# PQ-02: Star-Schema Warehouse

## 1. Objective

Build a separate DuckDB warehouse from the read-only PulseIQ source database,
validate table relationships and primary keys, and preserve source data.

## 2. Architecture

- Source database: `05_Starter/pulseiq.duckdb`
- Warehouse database: `05_Starter/pulseiq_warehouse.duckdb`
- Source schema: `raw`
- Warehouse schema: `warehouse`
- Source database remains unchanged.
- The warehouse contains 11 dimensions and 12 fact tables.

## 3. Star-schema relationship overview

The diagram shows the main analytical relationships. It is a logical overview;
not every relationship is a direct foreign-key constraint in DuckDB.

```mermaid
flowchart TD
    Date[dim_date]
    Employee[dim_employee]
    Department[dim_department]
    Role[dim_role]
    Location[dim_location]
    Account[dim_account]
    Client[dim_client]
    Vendor[dim_vendor]
    LeaveType[dim_leave_type]
    Team[dim_team]
    Sprint[dim_sprint]

    Payroll[fact_payroll]
    Attendance[fact_attendance_monthly]
    Leave[fact_leave]
    Headcount[fact_headcount_monthly]
    Forecast[fact_forecast]
    Ledger[fact_ledger]
    Expense[fact_expense]
    Invoice[fact_invoice]
    Payment[fact_invoice_payment]
    Task[fact_task]
    Anomaly[fact_anomaly]
    Attrition[fact_attrition_risk]

    Employee --> Payroll
    Department --> Payroll
    Date --> Payroll

    Employee --> Attendance
    Department --> Attendance
    Date --> Attendance

    Employee --> Leave
    LeaveType --> Leave
    Date --> Leave

    Department --> Headcount
    Date --> Headcount

    Department --> Forecast
    Date --> Forecast

    Account --> Ledger
    Department --> Ledger
    Date --> Ledger

    Account --> Expense
    Vendor --> Expense
    Department --> Expense
    Date --> Expense

    Client --> Invoice
    Account --> Invoice
    Date --> Invoice

    Invoice --> Payment
    Date --> Payment

    Team --> Task
    Sprint --> Task
    Employee --> Task

    Employee --> Anomaly
    Date --> Anomaly

    Employee --> Attrition
    Department --> Attrition
    Date --> Attrition

    Role --> Employee
    Department --> Employee
    Location --> Employee
    Department --> Role
    Department --> Team
    Account --> Vendor
    Account --> Account

_IncompleteInputError: incomplete input (514544361.py, line 6)

In [29]:
from pathlib import Path

docs_dir = PROJECT_ROOT / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

print("Documentation folder ready:", docs_dir)

Documentation folder ready: C:\Users\Hp\Documents\pulseiq\docs


In [30]:
schema_doc_path = docs_dir / "PQ-02_star_schema.md"

sections = [
    "# PQ-02: Star-Schema Warehouse\n\n",
    "## 1. Objective\n\n",
    "Build a separate DuckDB warehouse from the read-only PulseIQ source.\n\n",
    "## 2. Architecture\n\n",
    "- Source: `05_Starter/pulseiq.duckdb`\n",
    "- Warehouse: `05_Starter/pulseiq_warehouse.duckdb`\n",
    "- Source schema: `raw`\n",
    "- Warehouse schema: `warehouse`\n",
    "- Source database remains unchanged.\n\n",
    "## 3. Validation Results\n\n",
    "- 23 tables reconciled successfully.\n",
    "- 25 relationship checks passed.\n",
    "- 23 primary-key candidates passed validation.\n",
    "- 23 primary-key constraints were added and verified.\n",
    "- All 72 forecast date keys match the warehouse calendar.\n",
    "- Calendar extended by 90 days, through 2027-03-31.\n\n",
    "## 4. Known Exceptions\n\n",
    "- Payroll `PI-X0003`: employee `EMP-9001`, employee key `-1`.\n",
    "- Payroll `PI-X0004`: employee `EMP-9002`, employee key `-1`.\n",
    "- 12 company-wide forecast records use department ID `ALL`.\n",
    "- These records were retained rather than deleted or reassigned.\n\n",
    "## 5. Calendar Notes\n\n",
    "- Warehouse date rows: 2,647.\n",
    "- Source date rows: 2,557.\n",
    "- Generated holiday and working-day flags remain NULL because they were not verified.\n\n",
    "## 6. Primary Keys\n\n",
    "All 23 validated warehouse tables have primary-key constraints. "
    "`fact_headcount_monthly` uses the composite key `(date_key, department_id)`.\n\n",
    "## 7. Completion Status\n\n",
    "The documented PQ-02 integrity checks passed. Git status, commit, and pull-request "
    "steps must still be verified separately.\n"
]

schema_doc_path.write_text("".join(sections), encoding="utf-8")

print("Documentation created:", schema_doc_path)
print("File exists:", schema_doc_path.exists())
print("File size:", schema_doc_path.stat().st_size, "bytes")

Documentation created: C:\Users\Hp\Documents\pulseiq\docs\PQ-02_star_schema.md
File exists: True
File size: 1423 bytes


In [32]:
import subprocess

def git_command(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    print(result.stdout.strip() or "(no output)")
    if result.returncode != 0:
        print("Git message:", result.stderr.strip())
    return result

print("CURRENT BRANCH")
git_command("branch", "--show-current")

print("\nWORKING TREE STATUS")
git_command("status", "--short")

print("\nWAREHOUSE DATABASE IGNORE STATUS")
git_command(
    "check-ignore",
    "05_Starter/pulseiq_warehouse.duckdb"
)

CURRENT BRANCH
pq-02-star-schema

WORKING TREE STATUS
M PQ-01_ETL_Validation.ipynb
?? PQ-02_Star_Schema_Warehouse.ipynb
?? docs/PQ-02_star_schema.md

WAREHOUSE DATABASE IGNORE STATUS
05_Starter/pulseiq_warehouse.duckdb


CompletedProcess(args=['git', 'check-ignore', '05_Starter/pulseiq_warehouse.duckdb'], returncode=0, stdout='05_Starter/pulseiq_warehouse.duckdb\n', stderr='')

In [33]:
import subprocess

files_to_stage = [
    "PQ-02_Star_Schema_Warehouse.ipynb",
    "docs/PQ-02_star_schema.md",
]

result = subprocess.run(
    ["git", "add", "--", *files_to_stage],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

print("PQ-02 files staged successfully.")

print("\nSTAGED CHANGE SUMMARY")
subprocess.run(
    ["git", "diff", "--cached", "--stat"],
    cwd=PROJECT_ROOT,
    check=True
)

print("\nCURRENT STATUS")
subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    check=True
)

PQ-02 files staged successfully.

STAGED CHANGE SUMMARY

CURRENT STATUS


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [34]:
import subprocess

result = subprocess.run(
    ["git", "diff", "--cached", "--name-status"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True
)

print("STAGED FILES:")
print(result.stdout if result.stdout.strip() else "No files staged.")

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True
)

print("\nFULL GIT STATUS:")
print(status.stdout if status.stdout.strip() else "Working tree clean.")

STAGED FILES:
A	PQ-02_Star_Schema_Warehouse.ipynb
A	docs/PQ-02_star_schema.md


FULL GIT STATUS:
 M PQ-01_ETL_Validation.ipynb
AM PQ-02_Star_Schema_Warehouse.ipynb
A  docs/PQ-02_star_schema.md



In [ ]:
import subprocess

# Stage the latest saved version of the PQ-02 notebook
subprocess.run(
    ["git", "add", "--", "PQ-02_Star_Schema_Warehouse.ipynb"],
    cwd=PROJECT_ROOT,
    check=True
)

# Verify the staged files
result = subprocess.run(
    ["git", "diff", "--cached", "--name-status"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True
)

print("STAGED FILES:")
print(result.stdout)

# Confirm there are no unstaged changes to the PQ-02 notebook
result = subprocess.run(
    ["git", "diff", "--", "PQ-02_Star_Schema_Warehouse.ipynb"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True
)

print("Unstaged notebook changes:")
print(result.stdout if result.stdout else "None")